# Практикум 09 · Обработка потока ROS

Ниже находится полная глава курса, затем — ячейки практики. Прочитайте главу перед изменением параметров.

При **Run All** выполнение аппаратной части пропускается: `RUN_ON_ROBOT = False`. Включайте её после настройки окружения и собственных входных данных.


# 9.9. Собственный узел обработки изображения в ROS

**Работа на нашем роботе:** создайте [личную копию курса и выберите нужное окружение](../docs/SETUP_ROBOT.md). Все команды ниже выполняйте из её корня; результаты `outputs/` будут принадлежать вашей копии. Самостоятельный поиск камеры, контейнера и ROS-топиков разобран в той же инструкции.

[Назад: 9.8](../lessons/08_ros_image.md) · [Оглавление](../README.md) · [Далее: 9.10](../lessons/10_data.md)

**Результат:** поток бинарной маски красного объекта в отдельном ROS-топике, последний обработанный кадр и статистика. Дополнительно вы сравните собственный алгоритм с заводским пакетом `opencv_apps`. Ориентир: 60–90 минут.

## 1. Объясните путь сообщения

Драйвер камеры публикует `sensor_msgs/Image`. Наш узел подписывается на этот топик, получает BGR-массив через `cv_bridge`, выделяет красные пиксели в HSV и публикует новую одноканальную картинку. Заголовок исходного сообщения сохраняется: результат относится к тому же времени и системе координат.

В OpenCV для 8-битного HSV тон H занимает значения 0–179. Красный цвет находится у обоих концов этой шкалы, поэтому используются два диапазона H: 0–10 и 170–179. Условия S ≥ 100 и V ≥ 80 отсекают слишком ненасыщенные и тёмные пиксели. Эти числа — начальные параметры упражнения, которые проверяют при фактическом освещении.

## 2. Выберите ветку среды

Проверьте на роботе:

```bash
python3 -c "import rospy, cv_bridge; print('Python 3 ROS bridge OK')"
rostopic list
```

При доступном Python 3 мосте выполняйте пункты 3–4. Если у вашего Melodic работает только заводская Python 2 связка, выполняйте пункт 5. Это полноценный путь знакомства с обработкой ROS-изображений без пересборки системы. Подробная подготовка приведена в [инструкции настройки](../docs/SETUP_ROBOT.md).

## 3. Запустите собственную обработку

Подготовьте красный предмет и простой фон. Камера должна уже публиковать RGB-кадры. Подставьте подтверждённое имя несжатого топика:

```bash
rostopic type /camera/rgb/image_raw
python3 -m rosmaster_cv.lesson09_ros_processing --topic /camera/rgb/image_raw --seconds 30
```

В другом терминале, пока программа работает:

```bash
rostopic type /course/red_mask
rostopic echo -n 1 /course/red_mask/header
rosrun image_view image_view image:=/course/red_mask
```

Ожидаемый тип результата — `sensor_msgs/Image`, кодировка — `mono8`. Красный предмет станет белой областью, остальной фон — чёрной. Закройте окно просмотра после окончания упражнения. Работа собственного узла ограничена 30 секундами, а весь процесс, включая старт ROS, имеет дополнительный предел 40 секунд. При недоступном master программа завершается с диагностикой.

По завершении откройте `outputs/09/last_mask.png` и `outputs/09/metrics.json`. Поле `frames_processed` должно быть больше нуля. `last_red_fraction` — доля белых пикселей в последней маске. Доля 0.10 означает примерно 10% изображения; это ещё не площадь предмета в квадратных сантиметрах.

## 4. Проведите небольшой эксперимент

Выполните три запуска: красный предмет занимает маленькую область, затем большую, затем частично находится в тени. Для каждого запуска задайте отдельную папку:

```bash
python3 -m rosmaster_cv.lesson09_ros_processing --topic /camera/rgb/image_raw --seconds 15 --output-dir outputs/09/near
```

Сравните последнюю маску и долю красного. Если часть объекта исчезает в тени, посмотрите ограничения S и V в функции `red_mask` файла `rosmaster_cv/lesson09_ros_processing.py`. Измените один порог, повторите наблюдение и запишите, что произошло с объектом и фоном.

Наш выход ограничен пространством имён `/course/`. Для другого названия используйте, например, `--output-topic /course/team1/red_mask`. Вход и выход должны различаться: подписка на собственный результат привела бы к неправильной цепочке обработки.

## 5. Заводской маршрут opencv_apps

В проверенной главе Yahboom запуск подготовки изображения выглядит так:

```bash
roslaunch yahboomcar_visual opencv_apps.launch img_flip:=false
```

Этот launch подготавливает камеру и публикует `/image`. Не открывайте ту же камеру ещё одним драйвером. В отдельном терминале запустите один алгоритм:

```bash
roslaunch opencv_apps hsv_color_filter.launch
```

Проверьте вход `/image` в графе узлов. Настройки фильтра меняются через доступные параметры пакета и `rqt_reconfigure`. Затем остановите этот алгоритм Ctrl+C и сравните другой:

```bash
roslaunch opencv_apps edge_detection.launch
```

Наконец, отдельно попробуйте `roslaunch opencv_apps contour_moments.launch`. Одновременно запускайте только тот пример, который исследуете. Сохраните скриншоты результата и `rqt_graph`; запишите названия фактических входных и выходных топиков. Пакет в опубликованном образе находится в `~/software/library_ws/src/opencv_apps`.

## Задание и проверка

Сдайте три наблюдения с масками и таблицей порогов, освещения и доли выделенных пикселей. Для собственного узла приложите `metrics.json`; для заводской ветки — скриншоты трёх алгоритмов и графа. Объясните назначение двух красных диапазонов H, различие маски и исходного кадра, а также смысл сохранённого `header`. Критерий завершения — вы можете самостоятельно найти источник изображения, запустить обработку и показать полученный поток.

**Источники:** [Yahboom 9.9](https://www.yahboom.net/study/ROSMASTER-X3-PLUS/8150/), [официальный opencv_apps](https://github.com/ros-perception/opencv_apps), [OpenCV HSV/inRange](https://docs.opencv.org/4.x/da/d97/tutorial_threshold_inRange.html), [ROS REP-151](https://www.ros.org/reps/rep-0151.html).


---

## Ячейки практики

Запускайте последовательно. Команды используют тот же Python, что и ядро ноутбука. Рабочая папка дочерних процессов — корень репозитория. Ошибка команды останавливает ячейку и показывает причину.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Jupyter may start in the repository root or in its notebooks directory.
ROOT = None
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "rosmaster_cv").is_dir() and (candidate / "lessons").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или его папки notebooks.")

def run_module(module, *arguments):
    command = [sys.executable, "-m", "rosmaster_cv." + module]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

def run_tool(filename, *arguments):
    command = [sys.executable, str(ROOT / "tools" / filename)]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

print("Корень курса:", ROOT)
print("Python ядра:", sys.executable)


In [ ]:
RUN_ON_ROBOT = False
IMAGE_TOPIC = ""  # Впишите существующий топик цветного sensor_msgs/Image.
OUTPUT_TOPIC = "/course/red_mask"  # Топик результата этого упражнения.


In [ ]:
if RUN_ON_ROBOT:
    if not IMAGE_TOPIC or not IMAGE_TOPIC.startswith("/"):
        raise ValueError("Сначала укажите реальный IMAGE_TOPIC.")
    run_module("lesson09_ros_processing", "--topic", IMAGE_TOPIC,
               "--seconds", "30", "--output-topic", OUTPUT_TOPIC)
else:
    print("ROS-обработка выключена. Настройте IMAGE_TOPIC по инструкции главы перед включением.")


## Результат самостоятельной работы

Выполните задания и критерии приёмки из главы. Сохраните созданные изображения и JSON, параметры запуска и краткие ответы в своём отчёте. Повторите опыт с собственными данными после успешного учебного запуска.
